<a href="https://colab.research.google.com/github/minyi-k03/LargeLanguageModel/blob/Project-Based-Learning(PBL)/Llama3_2_vision_ChartQA_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Llama 3.2-Vision 모델 한국어 ChartQA 성능 테스트하기
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## Reference : https://huggingface.co/meta-llama/Llama-3.2-11B-Vision-Instruct
## ChartQA 데이터셋 : https://huggingface.co/datasets/ahmed-masry/ChartQA

In [ ]:
!nvidia-smi

# 라이브러리 설치

In [ ]:
# [Cell 1] 라이브러리 설치
import torch

# 1. GPU 확인
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("GPU가 없습니다. 런타임 유형을 T4로 변경하세요.")

print("\nInstalling Libraries for Llama-3.2 Vision...")

# 2. 필수 라이브러리 설치
!pip install -U "transformers>=4.45.0" "accelerate" "bitsandbytes" "huggingface_hub"


In [ ]:
!pip install datasets

# ChartQA 데이터셋 불러오기

In [ ]:
from datasets import load_dataset

# Hugging Face Hub에서 ChartQA 데이터셋 로드
dataset = load_dataset("ahmed-masry/ChartQA")

# 데이터셋의 정보 출력
print(dataset)

# 데이터셋의 특정 split 확인 (예: train, validation, test 등)
print(dataset['train'][0])  # 첫 번째 샘플 출력

In [ ]:
print(dataset['test'][0])  # 첫 번째 샘플 출력

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import io

test_image_1 = Image.open(io.BytesIO(dataset['test'][0]['image']))
test_image_1

In [ ]:
dataset['test'][0]['query']

In [ ]:
dataset['test'][0]['label']

# Llama 3.2 모델 불러오기

In [ ]:
# [Cell 2] Llama-3.2-11B-Vision 로드 (4-bit 양자화)
import os
import torch
from transformers import MllamaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# 1. 토큰 설정
os.environ['HF_TOKEN'] = "Input Your Token"

model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"

# 2. 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading Vision Model: {model_id}...")

# 3. 모델 로드 (Vision 모델 전용 클래스 사용)
model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config, # 4-bit 적용
    device_map="auto",
    torch_dtype=torch.float16
)

# 4. 프로세서 로드 (이미지+텍스트 처리용)
processor = AutoProcessor.from_pretrained(model_id)

print(f"Model Loaded on {model.device}")

In [ ]:
#정답이 틀
messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": dataset['test'][0]['query']}
    ]}
]
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(
    test_image_1,
    input_text,
    add_special_tokens=False,
    return_tensors="pt"
).to(model.device)

output = model.generate(**inputs, max_new_tokens=256)
print(processor.decode(output[0]))
# 정답 : 14

In [ ]:
test_image = Image.open(io.BytesIO(dataset['test'][1]['image']))
test_image

In [ ]:
query = dataset['test'][1]['query']
query

In [ ]:
label = dataset['test'][1]['label']
label

In [ ]:
#정답 맞춤
messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": query}
    ]}
]
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(
    test_image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt"
).to(model.device)

output = model.generate(**inputs, max_new_tokens=256)
print(processor.decode(output[0]))
# 정답 : 0.57

In [ ]:
dataset['test']

# 앞에 10개의 test data에 대해 테스트

In [ ]:
import matplotlib.pyplot as plt

for i in range(10):
    query = dataset['test'][i]['query']
    label = dataset['test'][i]['label']

    # Load the corresponding image
    test_image = Image.open(io.BytesIO(dataset['test'][i]['image']))

    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": query}
        ]}
    ]

    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(
        test_image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(**inputs, max_new_tokens=256)
    result = processor.decode(output[0])

    # Display the image
    plt.imshow(test_image)
    plt.axis('off')
    plt.title(f"Index: {i}")
    plt.show()

    print("------------------------------")
    print(f"Index: {i}")
    print(f"Query: {query}")
    print(f"Generated Output: {result}")
    print(f"Label (Ground Truth): {label}")

In [ ]:
# 정확도 분석
# 0. x
# 1. o
# 2. o
# 3. o
# 4. o
# 5. o
# 6. o
# 7. o
# 8. x
# 9. o
# 8/10 - 80%

# 한국어 쿼리에 대한 테스트

In [ ]:
korean_qeuries = [None] * 10
korean_qeuries[0] = "막대 그래프에 표시된 음식 항목은 몇 개입니까?"
korean_qeuries[1] = "양고기와 옥수수의 값 차이는 얼마입니까?"
korean_qeuries[2] = "차트에 표시된 막대의 개수는 몇 개입니까?"
korean_qeuries[3] = "마다가스카르의 총합 값이 피지보다 많습니까?"
korean_qeuries[4] = "가장 낮은 막대의 값은 얼마입니까?"
korean_qeuries[5] = "가장 높은 초록색 막대와 가장 낮은 초록색 막대의 차이는 얼마입니까?"
korean_qeuries[6] = "도널드 트럼프 대통령을 위험하다고 생각하는 사람의 비율은 몇 퍼센트입니까?"
korean_qeuries[7] = "'카리스마 있다'와 '대통령으로서 자격이 있다'의 퍼센트 합이 '강력한 지도자'보다 많습니까?"
korean_qeuries[8] = "네 번째로 인기 있었던 감정은 무엇이었습니까?"
korean_qeuries[9] = "우울감을 자주 느꼈던 사람보다 영감을 자주 느꼈던 사람이 몇 명 더 많습니까?"

In [ ]:
#Prompt Query 한국어로 변경
import matplotlib.pyplot as plt

for i in range(10):
    query = korean_qeuries[i]
    label = dataset['test'][i]['label']

    # Load the corresponding image
    test_image = Image.open(io.BytesIO(dataset['test'][i]['image']))

    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": query}
        ]}
    ]

    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(
        test_image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(**inputs, max_new_tokens=256)
    result = processor.decode(output[0])

    # Display the image
    plt.imshow(test_image)
    plt.axis('off')
    plt.title(f"Index: {i}")
    plt.show()

    print("------------------------------")
    print(f"Index: {i}")
    print(f"Query: {query}")
    print(f"Generated Output: {result}")
    print(f"Label (Ground Truth): {label}")

In [ ]:
# Query를 한국어로 변경하여 정확도 검사
# 0. x
# 1. x
# 2. o
# 3. o
# 4. o
# 5. x
# 6. o
# 7. o
# 8. x
# 9. x
# 5/10 - 50%